In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

movies = pd.read_csv("../data/movies.csv")
ratings = pd.read_csv("../data/ratings.csv")
links = pd.read_csv("../data/links.csv")
tags = pd.read_csv("../data/tags.csv")

In [15]:
# Understanding movies dataset

movies.head()


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [23]:
movies.info()


<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
dtypes: int64(1), str(2)
memory usage: 626.1 KB


In [24]:
movies.isnull().sum()

movieId    0
title      0
genres     0
dtype: int64

In [25]:
movies.duplicated().sum()

np.int64(0)

In [26]:
movies.describe(include="all")

,movieId,title,genres
count,9742.000000,9742,9742
unique,NaN,9737,951
top,NaN,Emma (1996),Drama
freq,NaN,2,1053
mean,42200.353623,NaN,NaN
std,52160.494854,NaN,NaN
min,1.000000,NaN,NaN
25%,3248.250000,NaN,NaN
50%,7300.000000,NaN,NaN
75%,76232.000000,NaN,NaN


In [27]:
# Understanding ratings dataset

ratings.head()
ratings.shape
ratings.info()
ratings.isnull().sum()
ratings.duplicated().sum()
ratings.describe()

<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


,userId,movieId,rating,timestamp
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,326.127564,19435.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,1.000000,1.000000,0.500000,8.281246e+08
25%,177.000000,1199.000000,3.000000,1.019124e+09
50%,325.000000,2991.000000,3.500000,1.186087e+09
75%,477.000000,8122.000000,4.000000,1.435994e+09
max,610.000000,193609.000000,5.000000,1.537799e+09


In [32]:
links.head()
links.shape
links.info()
links.isnull().sum()
links.duplicated().sum()


<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  9742 non-null   int64  
 1   imdbId   9742 non-null   int64  
 2   tmdbId   9734 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 228.5 KB


np.int64(0)

In [29]:
tags.head()


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [30]:
# so tag is basically a genre attached with each movieId by a userId

tags.shape
tags.info()
tags.isnull().sum()
tags.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   userId     3683 non-null   int64
 1   movieId    3683 non-null   int64
 2   tag        3683 non-null   str  
 3   timestamp  3683 non-null   int64
dtypes: int64(3), str(1)
memory usage: 151.7 KB


np.int64(0)

In [35]:
# TF-IDF Vectorization

from sklearn.feature_extraction.text import TfidfVectorizer
movies['features'] = movies['genres']

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies['features'])

tfidf_matrix.shape





(9742, 24)

In [36]:
tfidf.get_feature_names_out()

array(['action', 'adventure', 'animation', 'children', 'comedy', 'crime',
       'documentary', 'drama', 'fantasy', 'fi', 'film', 'genres',
       'horror', 'imax', 'listed', 'musical', 'mystery', 'no', 'noir',
       'romance', 'sci', 'thriller', 'war', 'western'], dtype=object)

In [38]:
# Cosine Similarity measures the similarity between two vectors by computing 
# the cosine of the angle between them.

from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

cosine_sim.shape



(9742, 9742)

In [43]:
# Create movie mapping

indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

In [47]:
# Lets validate 

movie_name = "Toy Story (1995)"
idx = indices[movie_name]

similarity_scores = cosine_sim[idx]

print(type(similarity_scores))
print(similarity_scores.shape)


<class 'numpy.ndarray'>
(9742,)


In [48]:
def recommend_movies(title):

    idx = indices[title]

    similarity_scores = list(enumerate(cosine_sim[idx]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:11]

    movie_indices = [i[0] for i in similarity_scores]

    return movies.iloc[movie_indices][['title', 'genres']]

In [49]:
recommend_movies("Toy Story (1995)")

,title,genres
1706,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy
2355,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
2809,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
3000,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
3568,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy
6194,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy
6486,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy
6948,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy
7760,Asterix and the Vikings (Astérix et les Viking...,Adventure|Animation|Children|Comedy|Fantasy
8219,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy


In [53]:
# Collaborative Filtering using SVD

# Instead of manually finding similar users or items, SVD learns:
# Hidden user preferences (latent factors)
# Hidden movie characteristics (latent factors)


from surprise import Dataset
from surprise import Reader
from surprise import SVD
from surprise.model_selection import train_test_split
from surprise import accuracy


reader = Reader(rating_scale=(0.5,5))

data = Dataset.load_from_df(
    ratings[['userId','movieId','rating']],
    reader
)

trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

model = SVD()

model.fit(trainset)

# Predict how User 1 would rate Movie 50
prediction = model.predict(
    uid=1,
    iid=50
)
# print(prediction)

predictions = model.test(testset)

accuracy.rmse(predictions)

RMSE: 0.8832


np.float64(0.883163614331889)

In [54]:
# recomendation function using SVD model

def recommend_for_user(user_id, n=10):

    watched = ratings[ratings['userId']==user_id]['movieId']

    unseen = movies[~movies['movieId'].isin(watched)]

    predictions = []

    for movie_id in unseen['movieId']:

        pred = model.predict(user_id, movie_id)

        predictions.append(
            (movie_id, pred.est)
        )

    predictions.sort(
        key=lambda x:x[1],
        reverse=True
    )

    top_movies = predictions[:n]

    movie_ids = [x[0] for x in top_movies]

    return movies[movies['movieId'].isin(movie_ids)][['title','genres']]


recommend_for_user(1)

,title,genres
266,Three Colors: Red (Trois couleurs: Rouge) (1994),Drama
474,Blade Runner (1982),Action|Sci-Fi|Thriller
602,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War
613,Trainspotting (1996),Comedy|Crime|Drama
686,Rear Window (1954),Mystery|Thriller
690,North by Northwest (1959),Action|Adventure|Mystery|Romance|Thriller
694,Casablanca (1942),Drama|Romance
896,One Flew Over the Cuckoo's Nest (1975),Drama
903,"Good, the Bad and the Ugly, The (Buono, il bru...",Action|Adventure|Western
910,Once Upon a Time in the West (C'era una volta ...,Action|Drama|Western


In [ ]:
# hybrid recommender system

def hybrid_recommend(user_id, movie_name):

    content = recommend_movies(movie_name)

    collab = recommend_for_user(user_id)

    final = pd.concat([content, collab])

    final = final.drop_duplicates(subset='title')

    return final.head(10)